## How to think about group operations


In [2]:
import numpy as np
import pandas as pd

df = pd.DataFrame(
    {
        "key1": ["a", "a", None, "b", "b", "a", None],
        "key2": pd.Series([1, 2, 1, 2, 1, None, 1], dtype="Int64"),
        "data1": np.random.standard_normal(7),
        "data2": np.random.standard_normal(7),
    }
)

df

,key1,key2,data1,data2
0,a,1,-0.887404,0.225624
1,a,2,0.977681,0.461724
2,None,1,-0.302590,-0.218110
3,b,2,0.399783,-0.393374
4,b,1,-1.402097,-1.132640
5,a,<NA>,-1.163029,-0.126293
6,None,1,-0.265851,-1.411814


In [3]:
# compute the mean of data1 using labels from key1:
grouped = df["data1"].groupby(df["key1"])  # same as df.groupby('key1').mean()['data1']
grouped

In [4]:
grouped.mean()

key1
a   -0.357584
b   -0.501157
Name: data1, dtype: float64

In [5]:
means = df["data1"].groupby([df["key1"], df["key2"]]).mean()
means

key1  key2
a     1      -0.887404
      2       0.977681
b     1      -1.402097
      2       0.399783
Name: data1, dtype: float64

In [6]:
means.unstack()

key2,1,2
key1,,
a,-0.887404,0.977681
b,-1.402097,0.399783


In [7]:
# get group sizes:
df.groupby(["key1", "key2"]).size()  # count() is similar

key1  key2
a     1       1
      2       1
b     1       1
      2       1
dtype: int64

### Iterating over groups


In [8]:
for name, group in df.groupby("key1"):  # for multiple keys, first value is a tuple
    print("name", name)
    print(group)

name a
  key1  key2     data1     data2
0    a     1 -0.887404  0.225624
1    a     2  0.977681  0.461724
5    a  <NA> -1.163029 -0.126293
name b
  key1  key2     data1     data2
3    b     2  0.399783 -0.393374
4    b     1 -1.402097 -1.132640


In [9]:
# compute a dictionary of the data pieces:
pieces = {name: group for name, group in df.groupby("key1")}
pieces["a"]

,key1,key2,data1,data2
0,a,1,-0.887404,0.225624
1,a,2,0.977681,0.461724
5,a,<NA>,-1.163029,-0.126293


In [10]:
# can group on other axes - example grouping by whether column starts with 'key' or 'data':
grouped = df.T.groupby({"key1": "key", "key2": "key", "data1": "data", "data2": "data"})
for key, values in grouped:
    print(key)
    print(values)

data
              0         1        2         3         4         5         6
data1 -0.887404  0.977681 -0.30259  0.399783 -1.402097 -1.163029 -0.265851
data2  0.225624  0.461724 -0.21811 -0.393374  -1.13264 -0.126293 -1.411814
key
      0  1     2  3  4     5     6
key1  a  a  None  b  b     a  None
key2  1  2     1  2  1  <NA>     1


### Selecting a column or subset of columns


In [11]:
df.groupby("key1")["data1"]  # equivalent to df['data1'].groupby(df['key1'])
df.groupby("key1")[["data2"]]  # equivalent to df[['data2']].groupby(df['key1'])

In [12]:
# compute mean but just for a single column:
df.groupby(["key1", "key2"])[["data2"]].mean()  # if we index ['data2'], return value is a Series instead of DataFrame

data2
key1 key2          
a    1     0.225624
     2     0.461724
b    1    -1.132640
     2    -0.393374

### Grouping with dictionaries and series


In [13]:
people = pd.DataFrame(
    np.random.standard_normal((5, 5)),
    columns=["a", "b", "c", "d", "e"],
    index=["Joe", "Steve", "Wanda", "Jill", "Trey"],
)
people.iloc[2:3, [1, 2]] = np.nan  # add some NAs

people

,a,b,c,d,e
Joe,0.667376,0.784165,-0.238332,-0.074217,1.017003
Steve,0.564215,-2.169531,-0.071374,-0.417004,1.341894
Wanda,1.208355,NaN,NaN,-0.491328,-0.910687
Jill,-0.075528,0.224287,-2.019130,1.162304,0.220224
Trey,-1.410112,0.325011,-0.454078,0.319977,-0.377258


In [14]:
# group correspondence for the columns - we wanna sum columns by group:
mapping = {"a": "red", "b": "red", "c": "blue", "d": "blue", "e": "red", "f": "orange"}

by_column = people.T.groupby(mapping)
by_column.sum()

,Joe,Steve,Wanda,Jill,Trey
blue,-0.312549,-0.488378,-0.491328,-0.856825,-0.134102
red,2.468543,-0.263422,0.297668,0.368983,-1.462358


In [15]:
map_series = pd.Series(mapping)
people.T.groupby(map_series).count()

,Joe,Steve,Wanda,Jill,Trey
blue,2,2,1,2,2
red,3,3,2,3,3


### Grouping with functions


In [16]:
# group by name length
people.groupby(len).sum()

,a,b,c,d,e
3,0.667376,0.784165,-0.238332,-0.074217,1.017003
4,-1.485640,0.549298,-2.473208,1.482281,-0.157034
5,1.772570,-2.169531,-0.071374,-0.908333,0.431207


In [17]:
# can also mix strategies (in this case, functions and arrays):
key_list = ["one", "one", "one", "two", "two"]
people.groupby([len, key_list]).min()

,,a,b,c,d,e
3,one,0.667376,0.784165,-0.238332,-0.074217,1.017003
4,two,-1.410112,0.224287,-2.019130,0.319977,-0.377258
5,one,0.564215,-2.169531,-0.071374,-0.491328,-0.910687


### Grouping by index levels


In [18]:
columns = pd.MultiIndex.from_arrays([["US", "US", "US", "JP", "JP"], [1, 3, 5, 1, 3]], names=["cty", "tenor"])

hier_df = pd.DataFrame(np.random.standard_normal((4, 5)), columns=columns)
hier_df

cty          US                            JP          
tenor         1         3         5         1         3
0      0.277647  0.303015 -1.611659  0.776028  0.807259
1      0.310712  1.102467 -0.857566  1.339501  0.415351
2     -0.083063 -0.738311 -0.987125  0.466844 -0.176917
3     -1.198591 -0.475962  0.600681 -0.409189 -0.931742

In [19]:
hier_df.groupby(level="cty", axis="columns").count()

C:\Users\biels\AppData\Local\Temp\ipykernel_2144\2795167867.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  hier_df.groupby(level="cty", axis="columns").count()


cty,JP,US
0,2,3
1,2,3
2,2,3
3,2,3


## Data aggregation

**Aggregation:** any data transformation that produces scalar values from arrays: `mean`, `count`, `min`, `sum`, etc.


In [20]:
df  # from chapter introduction

,key1,key2,data1,data2
0,a,1,-0.887404,0.225624
1,a,2,0.977681,0.461724
2,None,1,-0.302590,-0.218110
3,b,2,0.399783,-0.393374
4,b,1,-1.402097,-1.132640
5,a,<NA>,-1.163029,-0.126293
6,None,1,-0.265851,-1.411814


In [21]:
grouped = df.groupby("key1")
grouped["data1"].nsmallest(2)

key1   
a     5   -1.163029
      0   -0.887404
b     4   -1.402097
      3    0.399783
Name: data1, dtype: float64

In [22]:
# use a custom aggregation function (usually slower than native optimized ones):
def peak_to_peak(arr):
    return arr.max() - arr.min()


grouped.agg(peak_to_peak)

,key2,data1,data2
key1,,,
a,1,2.14071,0.588017
b,1,1.80188,0.739266


In [23]:
grouped.describe()  # works even though it's not an aggregation

key2                                           data1            ...  \
     count mean       std  min   25%  50%   75%  max count      mean  ...   
key1                                                                  ...   
a      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   3.0 -0.357584  ...   
b      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   2.0 -0.501157  ...   

                         data2                                          \
           75%       max count      mean       std       min       25%   
key1                                                                     
a     0.045138  0.977681   3.0  0.187018  0.295903 -0.126293  0.049665   
b    -0.050687  0.399783   2.0 -0.763007  0.522740 -1.132640 -0.947823   

                                    
           50%       75%       max  
key1                                
a     0.225624  0.343674  0.461724  
b    -0.763007 -0.578191 -0.393374  

[2 rows x 24 columns]

### Column-wise and multiple function application


In [24]:
tips = pd.read_csv("examples/tips.csv")
tips["tip_pct"] = tips["tip"] / tips["total_bill"]
tips.head()

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808


In [25]:
grouped = tips.groupby(["day", "smoker"])
grouped_pct = grouped["tip_pct"]
grouped_pct.agg("mean")

day   smoker
Fri   No        0.151650
      Yes       0.174783
Sat   No        0.158048
      Yes       0.147906
Sun   No        0.160113
      Yes       0.187250
Thur  No        0.160298
      Yes       0.163863
Name: tip_pct, dtype: float64

In [26]:
# applying a list of functions returns a DF with one column for each function:
grouped_pct.agg(["mean", "std", peak_to_peak])

mean       std  peak_to_peak
day  smoker                                  
Fri  No      0.151650  0.028123      0.067349
     Yes     0.174783  0.051293      0.159925
Sat  No      0.158048  0.039767      0.235193
     Yes     0.147906  0.061375      0.290095
Sun  No      0.160113  0.042347      0.193226
     Yes     0.187250  0.154134      0.644685
Thur No      0.160298  0.038774      0.193350
     Yes     0.163863  0.039389      0.151240

In [27]:
# apply same functions to two different columns:
functions = ["count", "mean", "max"]
result = grouped[["tip_pct", "total_bill"]].agg(functions)
result

tip_pct                     total_bill                  
              count      mean       max      count       mean    max
day  smoker                                                         
Fri  No           4  0.151650  0.187735          4  18.420000  22.75
     Yes         15  0.174783  0.263480         15  16.813333  40.17
Sat  No          45  0.158048  0.291990         45  19.661778  48.33
     Yes         42  0.147906  0.325733         42  21.276667  50.81
Sun  No          57  0.160113  0.252672         57  20.506667  48.17
     Yes         19  0.187250  0.710345         19  24.120000  45.35
Thur No          45  0.160298  0.266312         45  17.113111  41.19
     Yes         17  0.163863  0.241255         17  19.190588  43.11

In [28]:
ftuples = [("Average", "mean"), ("Variance", np.var)]  # custom column names
grouped[["tip_pct", "total_bill"]].agg(ftuples)

C:\Users\biels\AppData\Local\Temp\ipykernel_2144\4108143330.py:2: FutureWarning: The provided callable <function var at 0x0000019F59DE95A0> is currently using SeriesGroupBy.var. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "var" instead.
  grouped[["tip_pct", "total_bill"]].agg(ftuples)


tip_pct           total_bill            
              Average  Variance    Average    Variance
day  smoker                                           
Fri  No      0.151650  0.000791  18.420000   25.596333
     Yes     0.174783  0.002631  16.813333   82.562438
Sat  No      0.158048  0.001581  19.661778   79.908965
     Yes     0.147906  0.003767  21.276667  101.387535
Sun  No      0.160113  0.001793  20.506667   66.099980
     Yes     0.187250  0.023757  24.120000  109.046044
Thur No      0.160298  0.001503  17.113111   59.625081
     Yes     0.163863  0.001551  19.190588   69.808518

In [29]:
# different functions for each column, two for tip_pct, one of them with custom name, and one function for size:
grouped.agg({"tip_pct": [("Minimum", "min"), "max"], "size": "sum"})

tip_pct           size
              Minimum       max  sum
day  smoker                         
Fri  No      0.120385  0.187735    9
     Yes     0.103555  0.263480   31
Sat  No      0.056797  0.291990  115
     Yes     0.035638  0.325733  104
Sun  No      0.059447  0.252672  167
     Yes     0.065660  0.710345   49
Thur No      0.072961  0.266312  112
     Yes     0.090014  0.241255   40

### Returning aggregated data without row indexes


In [30]:
# day and smoker will now be columns of the result DFs instead of indexes
grouped = tips.groupby(["day", "smoker"], as_index=False)
grouped.mean(numeric_only=True)

# calling reset_index without specifying as_index would also work

,day,smoker,total_bill,tip,size,tip_pct
0,Fri,No,18.420000,2.812500,2.250000,0.151650
1,Fri,Yes,16.813333,2.714000,2.066667,0.174783
2,Sat,No,19.661778,3.102889,2.555556,0.158048
3,Sat,Yes,21.276667,2.875476,2.476190,0.147906
4,Sun,No,20.506667,3.167895,2.929825,0.160113
5,Sun,Yes,24.120000,3.516842,2.578947,0.187250
6,Thur,No,17.113111,2.673778,2.488889,0.160298
7,Thur,Yes,19.190588,3.030000,2.352941,0.163863


## `apply`: general split-apply-combine


In [31]:
# get n highest tip percentages
def top(df, n=5, column="tip_pct"):  # return value must be a df or a scalar
    return df.sort_values(column, ascending=False)[:n]


top(tips, n=6)

,total_bill,tip,smoker,day,time,size,tip_pct
172,7.25,5.15,Yes,Sun,Dinner,2,0.710345
178,9.60,4.00,Yes,Sun,Dinner,2,0.416667
67,3.07,1.00,Yes,Sat,Dinner,1,0.325733
232,11.61,3.39,No,Sat,Dinner,2,0.291990
183,23.17,6.50,Yes,Sun,Dinner,4,0.280535
109,14.31,4.00,Yes,Sat,Dinner,2,0.279525


In [32]:
# will apply to each group and concatenate them, creating a high-hierarchy index with the group label
tips.groupby("smoker").apply(top, n=3, column="total_bill")  # additional parameters will be forwarded to the function

C:\Users\biels\AppData\Local\Temp\ipykernel_2144\1889592082.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tips.groupby("smoker").apply(top, n=3, column="total_bill")  # additional parameters will be forwarded to the function


total_bill    tip smoker  day    time  size   tip_pct
smoker                                                           
No     212       48.33   9.00     No  Sat  Dinner     4  0.186220
       59        48.27   6.73     No  Sat  Dinner     4  0.139424
       156       48.17   5.00     No  Sun  Dinner     6  0.103799
Yes    170       50.81  10.00    Yes  Sat  Dinner     3  0.196812
       182       45.35   3.50    Yes  Sun  Dinner     3  0.077178
       102       44.30   2.50    Yes  Sat  Dinner     3  0.056433

In [33]:
# calling groups.describe is equivalent to:
def f(group):
    return group.describe()


grouped.apply(f)

C:\Users\biels\AppData\Local\Temp\ipykernel_2144\3204855179.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped.apply(f)


total_bill       tip  size   tip_pct
0 count    4.000000  4.000000  4.00  4.000000
  mean    18.420000  2.812500  2.25  0.151650
  std      5.059282  0.898494  0.50  0.028123
  min     12.460000  1.500000  2.00  0.120385
  25%     15.100000  2.625000  2.00  0.137239
...             ...       ...   ...       ...
7 min     10.340000  2.000000  2.00  0.090014
  25%     13.510000  2.000000  2.00  0.148038
  50%     16.470000  2.560000  2.00  0.153846
  75%     19.810000  4.000000  2.00  0.194837
  max     43.110000  5.000000  4.00  0.241255

[64 rows x 4 columns]

In [34]:
# suppressing group keys:
tips.groupby("smoker", group_keys=False).apply(top)

C:\Users\biels\AppData\Local\Temp\ipykernel_2144\3087596881.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tips.groupby("smoker", group_keys=False).apply(top)


,total_bill,tip,smoker,day,time,size,tip_pct
232,11.61,3.39,No,Sat,Dinner,2,0.291990
149,7.51,2.00,No,Thur,Lunch,2,0.266312
51,10.29,2.60,No,Sun,Dinner,2,0.252672
185,20.69,5.00,No,Sun,Dinner,5,0.241663
88,24.71,5.85,No,Thur,Lunch,2,0.236746
172,7.25,5.15,Yes,Sun,Dinner,2,0.710345
178,9.60,4.00,Yes,Sun,Dinner,2,0.416667
67,3.07,1.00,Yes,Sat,Dinner,1,0.325733
183,23.17,6.50,Yes,Sun,Dinner,4,0.280535
109,14.31,4.00,Yes,Sat,Dinner,2,0.279525


In [35]:
# quantile and bucket analysis
frame = pd.DataFrame({"data1": np.random.standard_normal(1000), "data2": np.random.standard_normal(1000)})
frame.head()

,data1,data2
0,0.489813,0.994850
1,1.186566,1.417721
2,-0.908657,-0.084612
3,0.316076,1.423255
4,0.540873,-1.719237


In [36]:
quartiles = pd.cut(frame["data1"], 4)
quartiles.head(10)

0     (0.168, 1.702]
1     (0.168, 1.702]
2    (-1.366, 0.168]
3     (0.168, 1.702]
4     (0.168, 1.702]
5    (-1.366, 0.168]
6    (-1.366, 0.168]
7     (1.702, 3.236]
8     (0.168, 1.702]
9     (0.168, 1.702]
Name: data1, dtype: category
Categories (4, interval[float64, right]): [(-2.906, -1.366] < (-1.366, 0.168] < (0.168, 1.702] < (1.702, 3.236]]

In [37]:
def get_stats(group):
    return pd.DataFrame({"min": group.min(), "max": group.max(), "count": group.count(), "mean": group.mean()})


grouped = frame.groupby(quartiles)
grouped.apply(get_stats)

C:\Users\biels\AppData\Local\Temp\ipykernel_2144\1910068442.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = frame.groupby(quartiles)


min       max  count      mean
data1                                                      
(-2.906, -1.366] data1 -2.900341 -1.371250     94 -1.743193
                 data2 -2.388339  1.892617     94  0.164775
(-1.366, 0.168]  data1 -1.338292  0.167210    466 -0.484384
                 data2 -3.545773  3.109580    466  0.014183
(0.168, 1.702]   data1  0.170029  1.685394    396  0.779807
                 data2 -2.855559  2.664360    396  0.005916
(1.702, 3.236]   data1  1.708820  3.235991     44  2.179164
                 data2 -1.774757  2.414342     44  0.067067

In [38]:
# the previous was a complicated version of:
grouped.agg(["min", "max", "count", "mean"]).stack(level=0)

C:\Users\biels\AppData\Local\Temp\ipykernel_2144\2099850329.py:2: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  grouped.agg(["min", "max", "count", "mean"]).stack(level=0)


min       max  count      mean
data1                                                      
(-2.906, -1.366] data1 -2.900341 -1.371250     94 -1.743193
                 data2 -2.388339  1.892617     94  0.164775
(-1.366, 0.168]  data1 -1.338292  0.167210    466 -0.484384
                 data2 -3.545773  3.109580    466  0.014183
(0.168, 1.702]   data1  0.170029  1.685394    396  0.779807
                 data2 -2.855559  2.664360    396  0.005916
(1.702, 3.236]   data1  1.708820  3.235991     44  2.179164
                 data2 -1.774757  2.414342     44  0.067067

In [39]:
# for equally-sized buckets:
quartiles_samp = pd.qcut(frame["data1"], 4, labels=False)
quartiles_samp.head()

grouped = frame.groupby(quartiles_samp)
grouped.apply(get_stats)

min       max  count      mean
data1                                           
0     data1 -2.900341 -0.679595    250 -1.280777
      data2 -2.615483  3.109580    250  0.033061
1     data1 -0.676766  0.013672    250 -0.300106
      data2 -3.210396  2.779089    250  0.085187
2     data1  0.013977  0.711114    250  0.354155
      data2 -3.545773  2.664360    250 -0.055829
3     data1  0.722522  3.235991    250  1.287142
      data2 -2.855559  2.414342    250  0.047149

### Example: filling missing values with group-specific values


In [40]:
states = ["Ohio", "New York", "Vermont", "Florida", "Oregon", "Nevada", "California", "Idaho"]
group_key = ["East", "East", "East", "East", "West", "West", "West", "West"]

data = pd.Series(np.random.standard_normal(8), index=states)
# removing some data:
data[["Vermont", "Nevada", "Idaho"]] = np.nan
data

Ohio          1.953672
New York      0.815803
Vermont            NaN
Florida      -0.410082
Oregon       -0.153497
Nevada             NaN
California    0.820964
Idaho              NaN
dtype: float64

In [41]:
data.groupby(group_key).mean()

East    0.786464
West    0.333734
dtype: float64

In [42]:
# fill NA values using group means
def fill_mean(group):
    return group.fillna(group.mean())


data.groupby(group_key).apply(fill_mean)

East  Ohio          1.953672
      New York      0.815803
      Vermont       0.786464
      Florida      -0.410082
West  Oregon       -0.153497
      Nevada        0.333734
      California    0.820964
      Idaho         0.333734
dtype: float64

In [43]:
# use group name to retrieve fill value:
fill_values = {"East": 0.5, "West": -1}


def fill_func(group):
    return group.fillna(fill_values[group.name])


data.groupby(group_key).apply(fill_func)

East  Ohio          1.953672
      New York      0.815803
      Vermont       0.500000
      Florida      -0.410082
West  Oregon       -0.153497
      Nevada       -1.000000
      California    0.820964
      Idaho        -1.000000
dtype: float64

### Example: random sampling and permutation

Draw random samples from a large dataset for Monte Carlo simulation.


In [44]:
suits = ["H", "S", "C", "D"]  # Hearts, Spades, Clubs, Diamonds
card_val = (list(range(1, 11)) + [10] * 3) * 4
base_names = ["A"] + list(range(2, 11)) + ["J", "K", "Q"]
cards = []
for suit in suits:
    cards.extend(str(num) + suit for num in base_names)

deck = pd.Series(card_val, index=cards)

In [45]:
def draw(deck, n=5):
    return deck.sample(n)


draw(deck)

4H      4
7H      7
10H    10
5C      5
10S    10
dtype: int64

In [46]:
# get two random cards from each suit - use a custom function to group:
def get_suit(card):
    return card[-1]  # the letter representing the suit


deck.groupby(get_suit).apply(draw, n=2)

C  6C      6
   8C      8
D  3D      3
   7D      7
H  5H      5
   10H    10
S  AS      1
   8S      8
dtype: int64

### Group weighted average and correlation

Operations between columns in a DF or two series.


In [47]:
df = pd.DataFrame(
    {
        "category": ["a", "a", "a", "a", "b", "b", "b", "b"],
        "data": np.random.standard_normal(8),
        "weights": np.random.uniform(size=8),
    }
)
df

,category,data,weights
0,a,-0.875501,0.526052
1,a,0.667957,0.663092
2,a,0.608396,0.637982
3,a,-0.976987,0.937715
4,b,1.786552,0.227876
5,b,0.456837,0.691601
6,b,1.195867,0.276576
7,b,0.980546,0.120018


In [48]:
# weighted average (my solution)
def weighted_avg(group):
    return (group["data"] * group["weights"]).sum() / (group["weights"].sum())


df.groupby("category").apply(weighted_avg)

C:\Users\biels\AppData\Local\Temp\ipykernel_2144\2093944756.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("category").apply(weighted_avg)


category
a   -0.197347
b    0.890144
dtype: float64

In [49]:
# solution from the book:
def get_wavg(group):
    return np.average(group["data"], weights=group["weights"])


df.groupby("category").apply(get_wavg)

C:\Users\biels\AppData\Local\Temp\ipykernel_2144\2356567718.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("category").apply(get_wavg)


category
a   -0.197347
b    0.890144
dtype: float64

In [ ]:
# correlate daily returns with SPX (S&P 500) index, grouping the correlation by year:
close_px = pd.read_csv("examples/stock_px.csv", parse_dates=True, index_col=0)
close_px.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2214 entries, 2003-01-02 to 2011-10-14
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    2214 non-null   float64
 1   MSFT    2214 non-null   float64
 2   XOM     2214 non-null   float64
 3   SPX     2214 non-null   float64
dtypes: float64(4)
memory usage: 86.5 KB


In [51]:
close_px.tail()

,AAPL,MSFT,XOM,SPX
2011-10-10,388.81,26.94,76.28,1194.89
2011-10-11,400.29,27.00,76.27,1195.54
2011-10-12,402.19,26.96,77.16,1207.25
2011-10-13,408.43,27.18,76.37,1203.66
2011-10-14,422.00,27.27,78.11,1224.58


In [ ]:
def spx_corr(group):
    return group.corrwith(group["SPX"])


rets = close_px.pct_change().dropna()
rets.tail()

,AAPL,MSFT,XOM,SPX
2011-10-10,0.051406,0.026286,0.036977,0.034125
2011-10-11,0.029526,0.002227,-0.000131,0.000544
2011-10-12,0.004747,-0.001481,0.011669,0.009795
2011-10-13,0.015515,0.008160,-0.010238,-0.002974
2011-10-14,0.033225,0.003311,0.022784,0.017380


In [55]:
by_year = rets.groupby(lambda r: r.year)
by_year.apply(spx_corr)

,AAPL,MSFT,XOM,SPX
2003,0.541124,0.745174,0.661265,1.0
2004,0.374283,0.588531,0.557742,1.0
2005,0.467540,0.562374,0.631010,1.0
2006,0.428267,0.406126,0.518514,1.0
2007,0.508118,0.658770,0.786264,1.0
2008,0.681434,0.804626,0.828303,1.0
2009,0.707103,0.654902,0.797921,1.0
2010,0.710105,0.730118,0.839057,1.0
2011,0.691931,0.800996,0.859975,1.0


In [ ]:
# annual correlation between apple and microsoft:
def cor_aapl_msft(group):
    return group["AAPL"].corr(group["MSFT"])


by_year.apply(cor_aapl_msft)

2003    0.480868
2004    0.259024
2005    0.300093
2006    0.161735
2007    0.417738
2008    0.611901
2009    0.432738
2010    0.571946
2011    0.581987
dtype: float64

### Group-wise linear regression


In [ ]:
import statsmodels.api as sm


def regress(data, yvar=None, xvars=None):
    Y = data[yvar]
    X = data[xvars]
    X["intercept"] = 1.0
    result = sm.OLS(Y, X).fit()
    return result.params


# yearly linear regression of AAPL on SPX:
by_year.apply(regress, yvar="AAPL", xvars=["SPX"])

,SPX,intercept
2003,1.195406,0.000710
2004,1.363463,0.004201
2005,1.766415,0.003246
2006,1.645496,0.000080
2007,1.198761,0.003438
2008,0.968016,-0.001110
2009,0.879103,0.002954
2010,1.052608,0.001261
2011,0.806605,0.001514


## Group transforms and "unwrapped" groupbys

`transform` is similar to `apply` but imposes more constraints on the function:

- can produce a scalar to be broadcast to the shape of the group
- can produce an object of the same shape as the input group
- must not mutate its input


In [66]:
df = pd.DataFrame({"key": ["a", "b", "c"] * 4, "value": np.arange(12.0)})
df

,key,value
0,a,0.0
1,b,1.0
2,c,2.0
3,a,3.0
4,b,4.0
5,c,5.0
6,a,6.0
7,b,7.0
8,c,8.0
9,a,9.0


In [ ]:
# group means by key:
g = df.groupby("key")["value"]
g.mean()

key
a    4.5
b    5.5
c    6.5
Name: value, dtype: float64

In [79]:
# series of the same shape as the value column, but with values replaced by the average grouped by key
g.transform(lambda group: group.mean())

0     4.5
1     5.5
2     6.5
3     4.5
4     5.5
5     6.5
6     4.5
7     5.5
8     6.5
9     4.5
10    5.5
11    6.5
Name: value, dtype: float64

In [ ]:
# same as:
g.transform("mean")

0     4.5
1     5.5
2     6.5
3     4.5
4     5.5
5     6.5
6     4.5
7     5.5
8     6.5
9     4.5
10    5.5
11    6.5
Name: value, dtype: float64

In [81]:
# multiply each group by 2 (function returns a series instead of a scalar):
g.transform(lambda group: group * 2)

0      0.0
1      2.0
2      4.0
3      6.0
4      8.0
5     10.0
6     12.0
7     14.0
8     16.0
9     18.0
10    20.0
11    22.0
Name: value, dtype: float64

In [82]:
# compute ranks for each group:
g.transform(lambda group: group.rank(ascending=False))

0     4.0
1     4.0
2     4.0
3     3.0
4     3.0
5     3.0
6     2.0
7     2.0
8     2.0
9     1.0
10    1.0
11    1.0
Name: value, dtype: float64

In [ ]:
# equivalents with apply and transform:
def normalize(x):
    return (x - x.mean()) / x.std()


g.transform(normalize)

0    -1.161895
1    -1.161895
2    -1.161895
3    -0.387298
4    -0.387298
5    -0.387298
6     0.387298
7     0.387298
8     0.387298
9     1.161895
10    1.161895
11    1.161895
Name: value, dtype: float64

In [84]:
g.apply(normalize)

key    
a    0    -1.161895
     3    -0.387298
     6     0.387298
     9     1.161895
b    1    -1.161895
     4    -0.387298
     7     0.387298
     10    1.161895
c    2    -1.161895
     5    -0.387298
     8     0.387298
     11    1.161895
Name: value, dtype: float64

In [ ]:
# fast path when used with transform - unwrapped group operation:
(df["value"] - g.transform("mean")) / g.transform("std")  # way faster than using custom normalize function

0    -1.161895
1    -1.161895
2    -1.161895
3    -0.387298
4    -0.387298
5    -0.387298
6     0.387298
7     0.387298
8     0.387298
9     1.161895
10    1.161895
11    1.161895
Name: value, dtype: float64

## Pivot tables and cross-tabulation


In [86]:
tips.head()

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808


In [ ]:
# margins=True adds partial totals ('All' column)
tips.pivot_table(
    index=["time", "day"], columns="smoker", values=["tip_pct", "size"], margins=True
)  # paraneter aggfunc allows to calculate other things

size                       tip_pct                    
smoker             No       Yes       All        No       Yes       All
time   day                                                             
Dinner Fri   2.000000  2.222222  2.166667  0.139622  0.165347  0.158916
       Sat   2.555556  2.476190  2.517241  0.158048  0.147906  0.153152
       Sun   2.929825  2.578947  2.842105  0.160113  0.187250  0.166897
       Thur  2.000000       NaN  2.000000  0.159744       NaN  0.159744
Lunch  Fri   3.000000  1.833333  2.000000  0.187735  0.188937  0.188765
       Thur  2.500000  2.352941  2.459016  0.160311  0.163863  0.161301
All          2.668874  2.408602  2.569672  0.159328  0.163196  0.160803

In [ ]:
# same output using groupby
tips.groupby(["time", "day", "smoker"]).mean(numeric_only=True)[["size", "tip_pct"]].unstack()

size             tip_pct          
smoker             No       Yes        No       Yes
time   day                                         
Dinner Fri   2.000000  2.222222  0.139622  0.165347
       Sat   2.555556  2.476190  0.158048  0.147906
       Sun   2.929825  2.578947  0.160113  0.187250
       Thur  2.000000       NaN  0.159744       NaN
Lunch  Fri   3.000000  1.833333  0.187735  0.188937
       Thur  2.500000  2.352941  0.160311  0.163863

In [99]:
# crosstab - a special case of pivot table that computes group frequencies:
data = """
Sample  Nationality  Handedness
1   USA  Right-handed
2   Japan    Left-handed
3   USA  Right-handed
4   Japan    Right-handed
5   Japan    Left-handed
6   Japan    Right-handed
7   USA  Right-handed
8   USA  Left-handed
9   Japan    Right-handed
10  USA  Right-handed
"""

from io import StringIO

data = pd.read_table(StringIO(data), sep="\s+")
data

,Sample,Nationality,Handedness
0,1,USA,Right-handed
1,2,Japan,Left-handed
2,3,USA,Right-handed
3,4,Japan,Right-handed
4,5,Japan,Left-handed
5,6,Japan,Right-handed
6,7,USA,Right-handed
7,8,USA,Left-handed
8,9,Japan,Right-handed
9,10,USA,Right-handed


In [ ]:
pd.crosstab(data["Nationality"], data["Handedness"], margins=True)

Handedness,Left-handed,Right-handed,All
Nationality,,,
Japan,2,3,5
USA,1,4,5
All,3,7,10
